# Experiment 5: SHAP Explainability Analysis (RQ4)

**Research Question**: How can Explainable AI (SHAP) translate the model's predictions into actionable, human-understandable insights for fraud analysts?

## Objectives
1. Generate global feature importance using SHAP values
2. Create local explanations for individual predictions (case studies)
3. Identify the most influential features for fraud detection

## Prerequisites
Run the experiment script first:
```bash
python -m src.experiments.exp5_shap experiment_name=shap-explainability-rq4
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

# Paths
SHAP_DIR = Path("../artifacts/shap_analysis")
print(f"Loading results from: {SHAP_DIR}")
print(f"Directory exists: {SHAP_DIR.exists()}")

## 1. Global Feature Importance

SHAP values show how much each feature contributes to predictions across all samples.

In [ ]:
# Load feature importance
importance_path = SHAP_DIR / "feature_importance.csv"
if importance_path.exists():
    importance_df = pd.read_csv(importance_path)
    print(f"Loaded {len(importance_df)} features")
    display(importance_df.head(20))
else:
    print(f"⚠️ Feature importance file not found. Run exp5_shap.py first.")

In [ ]:
# Display SHAP summary plots
summary_bar = SHAP_DIR / "shap_summary_bar.png"
if summary_bar.exists():
    print("SHAP Summary (Bar Plot)")
    display(Image(filename=str(summary_bar), width=800))
else:
    print("⚠️ Summary bar plot not found.")

In [ ]:
# Display SHAP beeswarm plot
summary_dot = SHAP_DIR / "shap_summary_dot.png"
if summary_dot.exists():
    print("SHAP Summary (Beeswarm Plot)")
    display(Image(filename=str(summary_dot), width=800))
else:
    print("⚠️ Summary dot plot not found.")

## 2. Feature Importance Analysis

In [ ]:
# Categorize features
if 'importance_df' in dir():
    # Define feature categories
    graph_features = ['listing_pagerank', 'listing_component_size', 'shared_contact_email_count',
                      'shared_ip_user_count', 'degree_total', 'is_isolated', 'user_listing_count']
    
    importance_df['category'] = importance_df['feature'].apply(
        lambda x: 'graph' if x in graph_features else 'tabular'
    )
    
    # Top features by category
    print("=" * 60)
    print("TOP FEATURES BY CATEGORY")
    print("=" * 60)
    
    for cat in ['graph', 'tabular']:
        print(f"\n{cat.upper()} FEATURES:")
        cat_df = importance_df[importance_df['category'] == cat].head(10)
        for _, row in cat_df.iterrows():
            print(f"  {row['feature']}: {row['importance']:.4f}")

## 3. Case Studies (Local Explanations)

Individual prediction explanations showing why specific listings were flagged as fraud.

In [ ]:
# Display case study waterfall plots
case_studies = list(SHAP_DIR.glob("shap_case_*_waterfall.png"))

if case_studies:
    print(f"Found {len(case_studies)} case studies")
    for case_path in sorted(case_studies)[:5]:  # Show first 5
        print(f"\n{case_path.stem}")
        display(Image(filename=str(case_path), width=700))
else:
    print("⚠️ No case study plots found.")

## 4. Key Findings Summary

In [ ]:
# Summary statistics
if 'importance_df' in dir():
    print("=" * 60)
    print("KEY FINDINGS")
    print("=" * 60)
    
    # Top 5 overall
    print("\nTop 5 Most Important Features:")
    for i, row in importance_df.head(5).iterrows():
        print(f"  {i+1}. {row['feature']} ({row['importance']:.4f})")
    
    # Graph feature contribution
    graph_importance = importance_df[importance_df['category'] == 'graph']['importance'].sum()
    total_importance = importance_df['importance'].sum()
    graph_pct = graph_importance / total_importance * 100
    
    print(f"\nGraph Feature Contribution: {graph_pct:.1f}% of total importance")
    print(f"Number of graph features in top 20: {len(importance_df[importance_df['category'] == 'graph'].head(20))}")

## 5. Actionable Insights for Fraud Analysts

Based on SHAP analysis, fraud analysts should prioritize:

1. **Graph-based red flags**: Low PageRank, isolated listings, shared contact info
2. **Behavioral patterns**: User listing count, IP sharing patterns
3. **Listing characteristics**: Specific features identified as high-importance

These insights can be used to:
- Create human-readable rules for manual review
- Prioritize investigation of high-risk listings
- Explain model decisions to stakeholders